In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Reddit-Slim") \
    .master("yarn") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 22:41:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/16 22:41:47 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark version: 3.5.8


In [2]:
df = spark.read.parquet("hdfs:///reddit/parquet/")

df_slim = df.select("content", "subreddit") \
            .filter(col("content").isNotNull()) \
            .filter(col("content") != "")

print(f"Row count: {df_slim.count()}")

[Stage 3:>                                                          (0 + 1) / 1]

Row count: 3848330


In [3]:
df_slim.write.mode("overwrite").parquet("hdfs:///reddit/slim/")
print("Done!")

Done!


In [4]:
import subprocess
result = subprocess.run(
    ["hdfs", "dfs", "-du", "-h", "/reddit/"],
    capture_output=True, text=True
)
print(result.stdout)

11.0 G   22.0 G   /reddit/parquet
373.8 M  747.6 M  /reddit/sample
3.3 G    6.5 G    /reddit/slim

